In [ ]:
try:
    from google.colab import drive
    drive.mount('/gdrive')
    dataset_root = '/gdrive/MyDrive/datasets'
    !pip install torchinfo tqdm
    colab = True
except Exception as e:
    print(e)
    print('Assuming we\'re not on colab.')
    dataset_root = './datasets'
    colab = False

print('Will store datasets in', dataset_root)

import os

if os.name == 'nt':
    print("Disabling multiprocessing because we're running on windows.")
    cpu_num = 0
elif colab:
    cpu_num = 2
else:
    cpu_num = os.cpu_count() // 2
    print('Dataloaders will use {} CPUs'.format(cpu_num))

In [ ]:
import torch
import torch.utils.data as tud
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary

import torchvision.datasets as tds
import torchvision.utils as tu

import matplotlib.pyplot as plt

The next cell will download and unpack MNIST, the classic digit recognition dataset. MNIST contains 60000 training images, and 10000 evaluation images across 10 digits (0...9) with equal amounts of each.

This dataset is split into a training set and an evaluation set. We only use the training set when training and updating the model, but we may use both when evaluating the quality of our training results. The evaluation set serves to inform of us how well the model generalizes, how well it learns a higher-level understanding of our data instead of just memorizing the data. Large models can contain enough parameters to simply memorize large portions of the dataset given enough time. You'll usually see the training loss continue to improve (go down) while the evaluation loss worsens (increases) when this happens. It's called overfitting.

In [ ]:
mnist_train = tds.MNIST(
    root=dataset_root,
    download=True,
    train=True,
)

mnist_eval = tds.MNIST(
    root=dataset_root,
    download=True,
    train=False,
)

PyTorch datasets can be accessed with the [] operator like a list. For the MNIST dataset, accessing a given index will return a tuple with the iamge in the first index, and the digit it represents as an integer in the second index.

In [ ]:
mnist_train[0]

As a first exercise and introduction to tensors, let's see what an "average" 1 looks like. We're doing to first use a conditional inside the [] operator to select all indicies with the given condition is true. This should give us all the images that represent a particular digit (in this case, 1). Then, we're going to stack them "vertically" as if you were stacking up a deck of cards, and then take the average through the stack to get a pixelwise average.

Sidenote: This cell also uses `ones.shape` or `ones.size()`. These can be used interchangeably. You'll find yourself dealing with tensors of 3 or 4 dimensions pretty frequently, and the first thing you should check when something goes wrong is whether the shape of the tensor you're working on is what you're expecting. It's easy to just add `print(mytensor.shape)` to your code to locate these kinds of bugs. 

In [ ]:
ones = mnist_train.data[mnist_train.targets == 1]
print(ones.size(), ones.shape)
sample = ones[0]
avg_one = ones.float().mean(dim=0)
print(avg_one.shape)
plt.imshow(torch.cat([avg_one, sample], dim=1))

Instead of trying to classify all 10 digits at once, we're going to first look at a reduced version of the problem that only needs to discrimate between two digits, turning this into a binary classification problem. The following cell wraps the mnist dataset into another dataset of only two classes, and provides `__getitem__` to support indexing, and `__len__` to support querying the number of image+label pairs in the dataset. In the process, we need to change the labels, giving 0 to one class, and 1 to the other. Now, when you index the dataset, you'll only get images of the two classes included here, and they'll either be labeled 0 or 1.

In [ ]:
class MnistBinary:
    def __init__(self, mnist, label0: int, label1: int):
        d0 = mnist.data[mnist.targets == label0]
        d1 = mnist.data[mnist.targets == label1]

        l0 = torch.zeros(len(d0))
        l1 = torch.ones(len(d1))
        
        self.data = torch.cat((d0, d1))
        self.targets = torch.cat((l0, l1))

    def __getitem__(self, index: int):
        return (self.data[index], self.targets[index])

    def __len__(self):
        return len(self.data)

    def random_grid(self, sz: int):
        samples_ix = torch.randint(low=0, high=len(self.data) - 1, size=(sz,))
        samples = self.data[samples_ix]
        samples = samples[:, None, ...]
        grid = tu.make_grid(samples)
        return grid.permute(1, 2, 0)        

In [ ]:
# The classes to use are selected here: 0 and 1
train = MnistBinary(mnist_train, 0, 1)
val = MnistBinary(mnist_eval, 0, 1)
plt.imshow(val.random_grid(64))

In [ ]:
batchsize = 32

train_loader = tud.DataLoader(train, batch_size=batchsize, num_workers=cpu_num, shuffle=True)
val_loader = tud.DataLoader(val, batch_size=batchsize, shuffle=True)

Now that we've seen the data, let's define a simple neural network composed of three linear layers: One input, one output, and one hidden.

This network will receive a batch of our 28x28 images, and then flatten them into a 1D array of size 784 (28*28). The final linear layer will produce a single scalar value that we will use as our prediction.

You may notice that there are no restrictions on what value our output layer, lin2, can produce, even though all of our labels are either 0 or 1. In the training loop below, we'll apply a sigmoid function to map the output range to be between 0 and 1, just like in the video.

In [ ]:
class MyNn(nn.Module):
    def __init__(self, in_sz, out_sz, hidden_sz=8):
        super().__init__()
        self.in_sz = in_sz
        self.out_sz = out_sz
        self.lin1 = nn.Linear(in_sz, hidden_sz)
        self.inner = nn.Linear(hidden_sz, hidden_sz)
        self.lin2 = nn.Linear(hidden_sz, out_sz)

    def forward(self, x):
        # This .view() implements the flattening operation.
        # -1 means to leave an index unchanged. The first index will be the batchsize.
        # Then, we want to flatten the remaining dimensions into one long dimension
        x = x.view(-1, self.in_sz)
        x = self.lin1(x)
        x = F.relu(x)
        x = self.inner(x)
        x = F.relu(x)
        x = self.lin2(x)
        return x
        

Side note: The cell below uses `unsqueeze(0)` and `squeeze()`. Unsqueeze adds dimensions of size 1 to a tensor at the requested index, and squeeze removes dimensions of size 1. You'll often find PyTorch functions that are designed to operate on tensors with a certain number of dimensions, so you can use squeeze and unsqueeze to make your data into the required shape.

In [ ]:
a = torch.zeros(1, 2,3)
print(a.shape, a.squeeze().shape, a.unsqueeze(2).shape)

In [ ]:
testmodel = MyNn(28 * 28, 1)
print(summary(testmodel))
test = train[0][0].unsqueeze(0).float()
testmodel(test)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = MyNn(28 * 28, 1, hidden_sz=8).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
lossfn = nn.L1Loss()
epochs = 10

for epoch in range(epochs):
    for i, (images, target) in enumerate(train_loader):
        optimizer.zero_grad()
        images = images.float().to(device)
        targets = target.float().to(device)

        outs = model(images)
        outs = F.sigmoid(outs)
        loss = lossfn(outs, targets.view(-1, 1))
        loss.backward()
        optimizer.step()

    losses = []
    for i, (images, target) in enumerate(val_loader):
        with torch.no_grad():
            images = images.float().to(device)
            targets = target.float().to(device)
            outs = model(images)
            outs = F.sigmoid(outs)
            loss = lossfn(outs, targets.view(-1, 1))
            losses.append(loss)
    print("Val loss {}".format(torch.Tensor(losses).mean()))

In [ ]:
total = len(val)
correct = 0
with torch.no_grad():
    for image, target in val:
        pred = model(image.float().to(device))
        pred = F.sigmoid(pred)
        if (pred.round() == target):
            correct += 1
            
    print("{:.2f}% correct".format(100*correct/total))

In [ ]:
from ipywidgets import interact

@interact(index=(0, len(val) - 1, 1))
def draw_preds(index=0):
    with torch.no_grad():
        image = val[index][0]
        pred = model(image.float().to(device))
        pred = F.sigmoid(pred)
        plt.imshow(image.float().cpu().squeeze(), cmap='gray')
        print(pred, pred.round())

How does such a simple neural network score >99% correct on binary classification? Well...

In [ ]:
with torch.no_grad():
    test = torch.zeros(28,28)
    test[14:16, 13:15] = 255
    plt.imshow(test, cmap='gray')
    pred = model(test.float().to(device))
    pred = pred.sigmoid()
    print(pred)

A 2x2 white square in the center quadrant is good enough to be classified as a 1. If you look at the training images, the majority of the 0's never have any white pixels in this region. The NN is able to exploit differences like this, and confidently predict 1 without learning a more complicated idea of what a 1 looks like.

What would our score be if we make a "model" that classifies based on those four pixels?

In [ ]:
total = len(val)
correct = 0
with torch.no_grad():
    for image, target in val:
        if image[14:16, 13:15].sum() > 100:
            pred = 1
        else:
            pred = 0
        if (pred == target):
            correct += 1
            
    print("{:.2f}% correct".format(100*correct/total))

# As homework...
As an exercise in tensor indexing, please produce an image that is the absolute value of the difference between the average of all the 1's (label 1) and all the 7's (label 7).
In English:
* Find the average 1 and average 7 like in the example
* Subtract one from the other
* Get the absolute value
* Plot it

Now, try the model on different pairs of numbers. Do you see meaningful differences in the loss value or accuracy scores for pairs that are visually similar (like 1 and 7) versus those that are very different (like 4 and 0)

Finally, write some code that locates one image in the "val" set that the model predicted the wrong label for and plot it.